In [0]:
# Databricks notebook source
# ==============================================================================
# PHASE 8: Silver Layer Transformation & Data Quality Cleaning
# ==============================================================================

storage_account_name = "datalakeseniorproj012026"
kv_scope_name = "kv-portfolio12026"

print("1. Authenticating & Injecting OAuth 2.0 configuration...")
client_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-id")
tenant_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-secret")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_uri = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_uri = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"

print("2. Reading immutable tables from Bronze Delta Lake...")
df_bronze_lineitem = spark.read.format("delta").load(bronze_uri + "lineitem/")
df_bronze_orders = spark.read.format("delta").load(bronze_uri + "orders/")

print("3. Applying Silver Data Quality & Cleaning Rules...")
from pyspark.sql.functions import col, trim

# Lineitem cleaning: Drop duplicates, ensure valid positive quantities/prices
df_silver_lineitem = df_bronze_lineitem \
    .dropDuplicates() \
    .filter(col("l_quantity") > 0) \
    .filter(col("l_extendedprice") >= 0)

# Orders cleaning: Drop duplicates, trim status text, ensure valid pricing
df_silver_orders = df_bronze_orders \
    .dropDuplicates() \
    .withColumn("o_orderstatus", trim(col("o_orderstatus"))) \
    .filter(col("o_totalprice") >= 0)

print("4. Writing cleaned datasets to Silver Delta Lake...")
df_silver_lineitem.write.format("delta").mode("overwrite").save(silver_uri + "lineitem/")
df_silver_orders.write.format("delta").mode("overwrite").save(silver_uri + "orders/")

print("===============================================================================")
print("SUCCESS: Silver layer cleaned, conformed, and persisted successfully.")
print("===============================================================================")

display(dbutils.fs.ls(silver_uri))